# Treinamento de um Agente de Blackjack com Q-Learning (e Contagem de Cartas)

## Objetivo
Este notebook implementa o treinamento de um agente para o jogo Blackjack utilizando o algoritmo **Q-Learning** integrado ao sistema de contagem de cartas Hi-Lo.

O objetivo é permitir que o agente aprenda por interação com o ambiente, descobrindo não só as melhores ações para as suas cartas, mas também identificando os momentos em que a contagem do baralho lhe é favorável.

## Papel desta etapa no projeto

Esta etapa introduz uma política aprendida com **aprendizado por reforço**. Com a introdução do baralho finito (*Shoe* de cassino) e do `true_count` no estado do jogo, a complexidade aumenta. Para evitar a Maldição da Dimensionalidade, agrupamos os valores de True Count para que o agente possa consolidar o aprendizado de forma eficiente.

In [1]:
import random
import pandas as pd
from utils_io import salvar_df

random.seed(42)

In [2]:
from blackjack_env import valor_mao, tem_as_utilizavel, iniciar_partida, step_blackjack, ShoeBlackjack

## Agrupamento do Estado (State Binning)

Para evitar que a Q-Table cresça indefinidamente com True Counts extremos, agrupamos a temperatura da mesa em grandes categorias. Isso acelera dramaticamente a convergência do modelo.

In [3]:
def agrupar_true_count(tc):
    """
    Agrupa o True Count para reduzir o espaço de estados da Q-Table.
    <= -2: Mesa Fria (ruim para o jogador)
    -1 a 1: Mesa Neutra
    >= 2: Mesa Quente (boa para o jogador)
    """
    if tc <= -2:
        return -2
    elif tc >= 2:
        return 2
    else:
        return tc

def estado_jogo_agrupado(mao_jogador, carta_dealer, true_count):
    return (
        valor_mao(mao_jogador),
        carta_dealer,
        int(tem_as_utilizavel(mao_jogador)),
        agrupar_true_count(true_count)
    )

## Funções auxiliares do algoritmo

As funções abaixo garantem a manutenção da Q-Table e a seleção de ações através da estratégia **epsilon-greedy**.

In [4]:
ACOES = ["hit", "stick"]

def garantir_estado(q_table, estado):
    if estado not in q_table:
        q_table[estado] = {"hit": 0.0, "stick": 0.0}

def melhor_acao(q_table, estado):
    garantir_estado(q_table, estado)

    q_hit = q_table[estado]["hit"]
    q_stick = q_table[estado]["stick"]

    if q_hit > q_stick:
        return "hit"
    elif q_stick > q_hit:
        return "stick"
    else:
        return random.choice(ACOES)

def escolher_acao_epsilon_greedy(q_table, estado, epsilon):
    garantir_estado(q_table, estado)

    if random.random() < epsilon:
        return random.choice(ACOES)
    return melhor_acao(q_table, estado)

## Hiperparâmetros do treinamento

Como o ambiente é complexo, aumentamos o treino para 1.000.000 de episódios, diminuímos a taxa de aprendizado e fizemos o epsilon decair de forma muito mais lenta.

In [5]:
alpha = 0.05
gamma = 1.0
epsilon_inicial = 1.0
epsilon_min = 0.05
epsilon_decay = 0.999995

n_episodios_treinamento = 1000000
n_episodios_avaliacao = 500000

## Treinamento do Agente

Instanciamos o baralho fora do ciclo de repetições. Dessa forma, as cartas vão escasseando e o *True Count* flutua como num casino real.

In [6]:
q_table = {}
historico_treinamento = []
epsilon = epsilon_inicial

# Instanciação do Shoe de Casino para o Treino
shoe_treino = ShoeBlackjack(num_baralhos=6)

for episodio in range(n_episodios_treinamento):
    mao_jogador, mao_dealer = iniciar_partida(shoe_treino)
    
    # Obtém o true count do momento e inicia o estado agrupado
    tc = shoe_treino.get_true_count()
    estado = estado_jogo_agrupado(mao_jogador, mao_dealer[0], tc)

    done = False
    passos = 0
    recompensa_final = 0
    resultado_final = "em_andamento"

    while not done:
        garantir_estado(q_table, estado)

        acao = escolher_acao_epsilon_greedy(q_table, estado, epsilon)

        # Passa o baralho para o ambiente
        _, recompensa, done, resultado, mao_jogador, mao_dealer = step_blackjack(
            mao_jogador, mao_dealer, acao, shoe_treino
        )

        q_atual = q_table[estado][acao]

        if done:
            target = recompensa
            proximo_estado = None
        else:
            # Recalcula o estado com a nova carta do jogador e nova contagem
            tc_novo = shoe_treino.get_true_count()
            proximo_estado = estado_jogo_agrupado(mao_jogador, mao_dealer[0], tc_novo)
            garantir_estado(q_table, proximo_estado)
            target = recompensa + gamma * max(q_table[proximo_estado].values())

        q_table[estado][acao] = q_atual + alpha * (target - q_atual)

        estado = proximo_estado if proximo_estado is not None else estado
        passos += 1
        recompensa_final = recompensa
        resultado_final = resultado

    # Gravamos o log apenas a cada 10 episódios para economizar memória (já que agora são 1 milhão)
    if episodio % 10 == 0:
        historico_treinamento.append({
            "id_episodio": episodio,
            "resultado": resultado_final,
            "recompensa_total": recompensa_final,
            "qtd_passos": passos,
            "epsilon": epsilon,
            "true_count_final": shoe_treino.get_true_count()
        })

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

In [7]:
df_treinamento = pd.DataFrame(historico_treinamento)
df_treinamento.head()

,id_episodio,resultado,recompensa_total,qtd_passos,epsilon,true_count_final
0,0,derrota,-1,1,1.00000,0
1,10,derrota,-1,1,0.99995,0
2,20,derrota,-1,3,0.99990,5
3,30,derrota,-1,1,0.99985,2
4,40,derrota,-1,1,0.99980,2


## Estrutura Final da Q-Table

A Q-Table agora exibe a coluna `true_count` variando estritamente entre os blocos agrupados (-2 a 2).

In [8]:
linhas_qtable = []

for estado, valores in q_table.items():
    soma_jogador, carta_dealer, as_utilizavel, true_count = estado

    q_hit = valores["hit"]
    q_stick = valores["stick"]

    if q_hit > q_stick:
        melhor = "hit"
    elif q_stick > q_hit:
        melhor = "stick"
    else:
        melhor = "empate"

    linhas_qtable.append({
        "soma_jogador": soma_jogador,
        "carta_dealer": carta_dealer,
        "as_utilizavel": as_utilizavel,
        "true_count": true_count,
        "q_hit": q_hit,
        "q_stick": q_stick,
        "melhor_acao": melhor
    })

df_qtable = pd.DataFrame(linhas_qtable).sort_values(
    by=["soma_jogador", "carta_dealer", "as_utilizavel", "true_count"]
).reset_index(drop=True)

df_qtable.head(20)

,soma_jogador,carta_dealer,as_utilizavel,true_count,q_hit,q_stick,melhor_acao
0,4,1,0,-2,-0.457239,-0.483503,hit
1,4,1,0,-1,-0.365463,-0.378942,hit
2,4,1,0,0,-0.436116,-0.517048,hit
3,4,1,0,1,-0.357939,-0.401373,hit
4,4,1,0,2,-0.401750,-0.536709,hit
5,4,2,0,-2,-0.120907,-0.196745,hit
6,4,2,0,-1,-0.097463,-0.135969,hit
7,4,2,0,0,-0.177535,-0.432622,hit
8,4,2,0,1,-0.131490,-0.205171,hit
9,4,2,0,2,-0.202403,-0.242699,hit


## Avaliação da política aprendida

O processo de avaliação submete o agente treinado a centenas de milhares de partidas reais num novo baralho contínuo.

In [9]:
def politica_qlearning(estado):
    garantir_estado(q_table, estado)
    return melhor_acao(q_table, estado)

In [10]:
def avaliar_politica_qlearning(n_episodios=5000):
    resultados = []
    shoe_aval = ShoeBlackjack(num_baralhos=6)

    for episodio in range(n_episodios):
        mao_jogador, mao_dealer = iniciar_partida(shoe_aval)
        carta_visivel_dealer = mao_dealer[0]

        done = False
        passos = 0
        resultado_final = "em_andamento"
        recompensa_final = 0

        while not done:
            # Utiliza a função com agrupamento para consultar a Q-Table
            tc = shoe_aval.get_true_count()
            estado = estado_jogo_agrupado(mao_jogador, carta_visivel_dealer, tc)
            acao = politica_qlearning(estado)

            _, recompensa, done, resultado, mao_jogador, mao_dealer = step_blackjack(
                mao_jogador, mao_dealer, acao, shoe_aval
            )

            passos += 1
            recompensa_final = recompensa
            resultado_final = resultado

        # Gravação otimizada (apenas 20% das partidas avaliadas para não criar arquivos gigantescos)
        if episodio % 5 == 0:
            resultados.append({
                "id_episodio": episodio,
                "politica": "qlearning",
                "resultado": resultado_final,
                "recompensa": recompensa_final,
                "total_jogador": valor_mao(mao_jogador),
                "total_dealer": valor_mao(mao_dealer),
                "dealer_carta_visivel": carta_visivel_dealer,
                "true_count_final": shoe_aval.get_true_count(),
                "qtd_passos": passos,
                "mao_jogador": str(mao_jogador),
                "mao_dealer": str(mao_dealer)
            })

    return pd.DataFrame(resultados)

df_resultados_qlearning = avaliar_politica_qlearning(n_episodios=n_episodios_avaliacao)
df_resultados_qlearning.head()

,id_episodio,politica,resultado,recompensa,total_jogador,total_dealer,dealer_carta_visivel,true_count_final,qtd_passos,mao_jogador,mao_dealer
0,0,qlearning,vitoria,1,12,22,5,0,2,"[2, 1, 9]","[5, 10, 7]"
1,5,qlearning,derrota,-1,15,21,3,0,1,"[10, 5]","[3, 10, 8]"
2,10,qlearning,derrota,-1,16,19,10,1,1,"[6, 10]","[10, 9]"
3,15,qlearning,derrota,-1,15,17,10,2,1,"[10, 5]","[10, 3, 4]"
4,20,qlearning,vitoria,1,21,23,9,4,2,"[4, 10, 7]","[9, 6, 8]"


In [11]:
taxa_vitoria = (df_resultados_qlearning["resultado"] == "vitoria").mean()
taxa_derrota = (df_resultados_qlearning["resultado"] == "derrota").mean()
taxa_empate = (df_resultados_qlearning["resultado"] == "empate").mean()

df_metricas_qlearning = pd.DataFrame([{
    "politica": "qlearning",
    "episodios_treinamento": n_episodios_treinamento,
    "episodios_avaliacao": n_episodios_avaliacao,
    "alpha": alpha,
    "gamma": gamma,
    "epsilon_inicial": epsilon_inicial,
    "epsilon_final": epsilon,
    "epsilon_min": epsilon_min,
    "epsilon_decay": epsilon_decay,
    "recompensa_media_treinamento": df_treinamento["recompensa_total"].mean(),
    "recompensa_media_avaliacao": df_resultados_qlearning["recompensa"].mean(),
    "taxa_vitoria_avaliacao": taxa_vitoria,
    "taxa_derrota_avaliacao": taxa_derrota,
    "taxa_empate_avaliacao": taxa_empate,
    "passos_medios_avaliacao": df_resultados_qlearning["qtd_passos"].mean()
}])

df_metricas_qlearning

,politica,episodios_treinamento,episodios_avaliacao,alpha,gamma,epsilon_inicial,epsilon_final,epsilon_min,epsilon_decay,recompensa_media_treinamento,recompensa_media_avaliacao,taxa_vitoria_avaliacao,taxa_derrota_avaliacao,taxa_empate_avaliacao,passos_medios_avaliacao
0,qlearning,1000000,500000,0.05,1.0,1.0,0.05,0.05,0.999995,-0.13741,-0.0555,0.42542,0.48092,0.09366,1.57705


In [12]:
df_resultados_qlearning["resultado"].value_counts()

resultado
derrota    48092
vitoria    42542
empate      9366
Name: count, dtype: int64

## Persistência dos artefatos

Guardamos os ficheiros que alimentarão a análise exploratória. Os ficheiros continuam com o sufixo `_hilo`.

In [13]:
salvar_df(df_qtable, "blackjack_q_table_hilo", pasta="modelos")
salvar_df(df_treinamento, "blackjack_treinamento_qlearning_hilo", pasta="dados")
salvar_df(df_resultados_qlearning, "blackjack_resultados_qlearning_hilo", pasta="dados")
salvar_df(df_metricas_qlearning, "blackjack_metricas_qlearning_hilo", pasta="resultados")

Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\modelos\blackjack_q_table_hilo.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\dados\blackjack_treinamento_qlearning_hilo.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\dados\blackjack_resultados_qlearning_hilo.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\resultados\blackjack_metricas_qlearning_hilo.csv
